# Lab: CORTEX AI Functions Part 1

📚 In this lab you will learn and practice the following:

❄️ Use task-based AI functions: SENTIMENT, AI_SENTIMENT, AI_TRANSLATE, AI_SUMMARIZE_AGG, AI_CLASSIFY

❄️ Generate embeddings and compute similarity scores using AI_EMBED and AI_SIMILARITY

❄️ Call Cortex AI functions from Snowpark Python

❄️ Use AI functions with CTEs

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps.  In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput). 

If you find your queries are running for more than 5 minutes, cancel and come back and try them later. 



📌 Note:

Your results may differ slightly from those displayed in the lab files and workbook, but this variation is normal.

- **Data quality matters**: Assess the quality of your data before running AI functions - poor input produces poor output.
- **Sensitive data**: Be aware of any PII or sensitive content you pass to these functions. Governance controls are covered in the Security and Governance module.
- **Tokens**: The unit of cost for LLM functions. Estimate: words × 1.5 ≈ token count. See the [Snowflake Credit Consumption Table](https://www.snowflake.com/legal-files/CreditConsumptionTable.pdf) for pricing.

---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL statements. Select any query and ask *"Explain this SQL"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any function? Ask CoCo *"What does [function name] do?"* to get its syntax, supported options, and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.

---

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

## Introduction

Snowflake Cortex AI functions are built-in SQL functions that bring AI capabilities directly to your data. Snowflake's AI functions are easy to use, just like regular functions. You simply provide the necessary inputs, and they return the output. Snowflake uses a variety of models from different companies, including Anthropic, Mistral, Meta, Google, OpenAI, Deepseek, and Snowflake arctic-embed, and will update them to ensure the best performance. Most functions do not require you to specify a model. Functions such as AI_COMPLETE and AI_EMBED can be called from a REST api for latency-sensitive use cases.

| Function | What it does |
| :--- | :--- |
| `SENTIMENT` | Returns a -1 to 1 sentiment score for English text |
| `AI_SENTIMENT` | Returns structured, category-level sentiment analysis |
| `AI_TRANSLATE` | Translates text between 14 supported languages |
| `AI_SUMMARIZE_AGG` | Aggregate summarization; handles datasets larger than a single context window |
| `AI_CLASSIFY` | Classifies text into user-defined categories |
| `AI_EMBED` | Generates an embedding vector for text or image input |
| `AI_SIMILARITY` | Computes a similarity score between two text or image inputs |

 **AI_COMPLETE and TRY_COMPLETE** → covered in Cortex AI Functions Part 2

 **AI_COMPLETE for images,AI_TRANSCRIBE, AI_FILTER** → covered in AI Functions Part 3
 
 **AI_PARSE_DOCUMENT, AI_EXTRACT** → covered in the Intelligent Document Processing 
 
 **AI_REDACT** → covered in the Security and Governance 

### Set up your current context for the role, database, schema and warehouse.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_DB'
print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r Setup_Context_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA raw;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: LLM Functions Part 1';
SHOW PARAMETERS LIKE 'query_tag' in session 
->> SELECT "value" AS query_tag FROM $1;


## SENTIMENT


 
In your **{{user}}_GENAI_DB**, the raw schema contains a review table from traveller reviews of activities. We can run the SENTIMENT analysis against these reviews to see what the traveller sentiment is like with a simple SELECT statement. In the past, we needed complex machine learning models to do similar analysis. Watch what happens when you run the next statement.

Returns a sentiment score from **-1 to 1** for English-language text. Negative values indicate negative sentiment, positive values indicate positive, and values near 0 are neutral.

### SENTIMENT function use cases.

| Industry | Use Case | Business Value |
| :--- | :--- | :--- |
| Retail & E-commerce 🛍️ | **Customer Experience Management** | **Provides rapid, at-scale insights** into customer satisfaction to improve products and boost retention. |
| Marketing & PR 📢 | **Brand & Campaign Monitoring** | Enables **proactive brand reputation management** and measures marketing ROI by analyzing public reaction. |
| Human Resources (HR) 👩‍💼 | **Employee Engagement Analysis** | Offers an **aggregated view of employee morale**, helping to reduce churn and improve workplace culture. |

### Run sentiment on sample text.

Run the function against a negative review, a positive one, and a neutral phrase to observe the score range - then apply it to the full review table.

In [ ]:
%%sql -r Sentiment_Negative_sql
-- Run SENTIMENT on a hardcoded negative review to see a score near -1
SELECT snowflake.cortex.SENTIMENT('Ghastly experience. I am done with that company. I am taking legal action against them. Never going to do business with them again.') AS sentiment_grading;

In [ ]:
%%sql -r Sentiment_Positive_sql
-- Store a positive review in a variable and score its sentiment (expect near +1)
SET input_text = 'I loved the snowcapped mountains. It was 100% the best experience I have ever had.';

SELECT snowflake.cortex.SENTIMENT($input_text) AS sentiment_grading;

In [ ]:
%%sql -r Sentiment_Neutral_sql
-- Score neutral text and label it using a CASE expression with thresholds
SET input_text = 'He wore a coat';

SELECT snowflake.cortex.SENTIMENT($input_text) AS sentiment_grading,
    CASE
        WHEN (sentiment_grading >= 0.075) THEN 'POSITIVE'
        WHEN (sentiment_grading <= -0.075) THEN 'NEGATIVE'
        ELSE 'NEUTRAL'
    END as sentiment_label;

### Using the sentiment function for textual analysis.

Run SENTIMENT across all rows in the review table to classify traveler sentiment at scale.

In [ ]:
%%sql -r Sentiment_Table_sql
-- Apply SENTIMENT to every row in the review table and label each one
SELECT snowflake.cortex.SENTIMENT(review_text) AS sentiment_grading,
    CASE
        WHEN (sentiment_grading >= 0.075) THEN 'POSITIVE'
        WHEN (sentiment_grading <= -0.075) THEN 'NEGATIVE'
        ELSE 'NEUTRAL'
    END as sentiment_label
FROM {{user}}_genai_db.raw.review

In [ ]:
%%sql -r Sentiment_Aggregate_sql
-- Count how many reviews fall into each sentiment category (POSITIVE, NEGATIVE, NEUTRAL)
WITH sentiment AS (
    SELECT 
        snowflake.cortex.SENTIMENT(review_text) AS sentiment_grading,
        CASE
            WHEN sentiment_grading >= 0.075 THEN 'POSITIVE'
            WHEN sentiment_grading <= -0.075 THEN 'NEGATIVE'
            ELSE 'NEUTRAL'
        END AS sentiment_label
    FROM {{user}}_genai_db.raw.review
)

SELECT 
    sentiment_label, 
    COUNT(*) AS review_classification_count
FROM sentiment
GROUP BY sentiment_label;


## AI_SENTIMENT

Analyzes sentiment across 7 languages with optional category-level breakdown. Specify up to 10 aspects (max 30 chars each) and get structured results classified as positive, negative, neutral, mixed, or unknown.

### Getting started with AI_SENTIMENT.

Run AI_SENTIMENT with and without a categories array to see how the structured output compares to the basic SENTIMENT score.

In [ ]:
%%sql -r AI_Sentiment_Basic_sql
-- Overall sentiment
SELECT AI_SENTIMENT('The hotel was amazing with great service, but the food was disappointing and overpriced') as hotel_sentiment;

### Analyze reviews with specific categories.

Run AI_SENTIMENT with `['service', 'cost', 'quality', 'booking', 'location']` against traveler reviews to get per-aspect sentiment.

In [ ]:
%%sql -r AI_Sentiment_Categories_sql
-- Analyze traveler reviews with specific categories
SELECT
    review_text,
   AI_SENTIMENT(
        review_text,
        ['service', 'cost', 'quality', 'booking', 'location']
    ) AS categorized_sentiment
FROM {{user}}_genai_db.transformed.review
LIMIT 5;


### Extracting specific sentiment data.

Use JSON path notation to pull individual category sentiment values from the AI_SENTIMENT result - useful for filtering and aggregating by a specific aspect.

In [ ]:
%%sql -r AI_Sentiment_Extract_sql
-- Extract and filter sentiment by specific categories
WITH sentiment_analysis AS (
    SELECT 
        review_text,
        AI_SENTIMENT(
            review_text, 
            ['service', 'cost', 'quality']
        ) AS sentiment_result
    FROM {{user}}_genai_db.transformed.review
    LIMIT 10
)
SELECT 
    review_text,
    sentiment_result:categories[0]:sentiment::STRING AS overall_sentiment,
    sentiment_result:categories[1]:sentiment::STRING AS service_sentiment,
    sentiment_result:categories[2]:sentiment::STRING AS cost_sentiment,
    sentiment_result:categories[3]:sentiment::STRING AS quality_sentiment
FROM sentiment_analysis
WHERE sentiment_result:categories[0]:sentiment::STRING != 'unknown';


## AI_TRANSLATE

Translates text from any supported language to any other. Pass an empty string `''` as the source code to enable automatic language detection.

### TRANSLATE function use cases.

| Use Case | Industry | Business Value |
| :--- | :--- | :--- |
| 🎧 **Global Customer Support** | Technology & SaaS | Translate **support tickets** into a single language for a centralized team, improving response times and efficiency. |
| 🌐 **International Market Research** | E-commerce & Retail | Translate **foreign product reviews** and social media to gain a unified, actionable view of global market trends. |
| 🏢 **Multinational Communications** | Manufacturing & Logistics | Translate **internal documents and training materials** to ensure consistent messaging across a **multilingual workforce**. |

### Getting started with the translate function.

Translate a sample phrase to Swedish, then Korean, then back to English to verify the round-trip - finally, use auto-detection to translate Spanish text without specifying the source language.

In [ ]:
%%sql -r Translate_Set_Text_sql
-- Set a source text variable to translate in subsequent cells
SET source_text = 'trail was rugged and uneven, not suitable for beginners. ';

In [ ]:
%%sql -r Translate_Swedish_sql
-- Translate the English source text to Swedish
SELECT AI_TRANSLATE($source_text, 'en', 'sv') AS translated_to_swedish;

In [ ]:
%%sql -r Translate_Korean_sql
-- Translate the English source text to Korean
SELECT AI_TRANSLATE($source_text, 'en', 'tr') AS translated_to_korean;

In [ ]:
%%sql -r Translate_Roundtrip_sql
-- Round-trip translation: English → Korean → English to test translation fidelity
WITH to_korean AS (
    SELECT AI_TRANSLATE($source_text, 'en', 'ko') AS translated_to_korean
),
back_to_english AS (
    SELECT AI_TRANSLATE(translated_to_korean, 'ko', 'en') AS translated_back_to_english
    FROM to_korean
)
SELECT translated_back_to_english
FROM back_to_english;

In [ ]:
%%sql -r Translate_Auto_Detect_sql
-- Auto-detect the source language (empty string) and translate to English
SET source_text = 'Trae una chaqueta y zapatos resistentes.';
SELECT AI_TRANSLATE($source_text, '', 'en') AS translated_to_english;

## AI_SUMMARIZE_AGG

An **aggregate function** that summarizes one or more text inputs into a concise synopsis. Unlike scalar functions, it can handle datasets larger than a single model's context window.

### AI_SUMMARIZE_AGG use cases.
| Industry | Use Case | Business Value |
| :--- | :--- | :--- |
| Retail & E-commerce | 🗣️ **Customer Feedback Synthesis** | Distill thousands of reviews into **key themes** to quickly identify product issues and prioritize improvements. |
| Financial Services | 📈 **Competitive Market Analysis** | Aggregate competitor earnings call transcripts to rapidly identify **strategic shifts and industry trends**. |
| Legal & Compliance | ⚖️ **Contract & Discovery Review** | Summarize large volumes of legal documents to quickly extract **key clauses and precedents, accelerating review**. |

### Getting started with AI_SUMMARIZE_AGG.

Create a temporary table with balloon history text, run AI_SUMMARIZE_AGG to produce a short synopsis, then pass a URL string to observe how the function treats it as literal text rather than fetching its content.

In [ ]:
%%sql -r Summarize_Create_Table_sql

USE SCHEMA {{user}}_genai_db.raw;

CREATE OR REPLACE TEMPORARY TABLE balloon_technical (text STRING);


INSERT INTO  balloon_technical VALUES 
('On November 21, 1783, the first recorded manned flight in a hot air balloon took place in Paris. The balloon, made from paper and silk by the Montgolfier brothers, was piloted by two noblemen from the court of Louis XVI and Marie Antoinette. The 22-minute flight ascended 500 feet above the rooftops of Paris before landing miles away in the vineyards. Local farmers, suspicious of this "fiery dragon" from the sky, were offered champagne by the pilots to calm their fears - a tradition that continues in ballooning to this day.
Declared the "sport of the Gods" by Marie Antoinette, ballooning, especially gas ballooning, soon became a popular activity across Europe. On January 19, 1784, in Lyon, France, Joseph Montgolfier made his only recorded flight in one of the largest balloons ever built. On September 15, 1784, Italian Vincenzo Lunardi completed the first balloon flight outside France, launching from Moorfields, England, and landing near Ware. On November 30, 1784, Frenchman Jean-Pierre Blanchard and American John Jeffries launched from Rhedarium Garden, London, making their first flight. On January 7, 1785, the same duo became the first to successfully fly across the English Channel.On January 9, 1793, Jean-Pierre Blanchard piloted the first balloon flight in North America, which took place in Philadelphia.Ballooning quickly captivated Europe and laid the foundation for modern aviation');



In [ ]:
%%sql -r Summarize_Basic_sql
SELECT AI_SUMMARIZE_AGG(text) AS summary, 
ARRAY_SIZE(SPLIT(AI_SUMMARIZE_AGG(text), ' ')) AS word_count
FROM balloon_technical;

In [ ]:
%%sql -r Summarize_URL_sql
SELECT AI_SUMMARIZE_AGG(
    'https://quickstarts.snowflake.com/guide/
    getting_started_with_snowpark_in_snowflake_python_worksheets/index.html'
) AS summary;

### Summarize all reviews for each activity.

Use AI_SUMMARIZE_AGG as an aggregate function with GROUP BY to produce a single summary per activity. This shows the real power of the function - condensing many reviews into one concise overview.

In [ ]:
%%sql -r Summarize_Long_Docs_sql
-- Summarize all reviews per activity using AI_SUMMARIZE_AGG as an aggregate function
SELECT
  a.name AS activity_name,
  COUNT(r.review_text) AS review_count,
  AI_SUMMARIZE_AGG(r.review_text) AS review_summary
FROM
  {{user}}_genai_db.raw.review r
  JOIN {{user}}_genai_db.raw.booking b ON r.booking_id = b.booking_id
  JOIN {{user}}_genai_db.raw.activity a ON b.activity_id = a.activity_id
GROUP BY
  a.name
ORDER BY
  review_count DESC
LIMIT 3;

In [ ]:
import textwrap

# Convert the SQL result to a pandas DataFrame and display the first summary wrapped at 80 chars
df = Summarize_Long_Docs_sql.to_pandas()
run_output = df["REVIEW_SUMMARY"].iloc[0]
wrapped_text = "\n".join(textwrap.wrap(run_output, width=80))
print(wrapped_text)

### Summarize multiple documents.

Run AI_SUMMARIZE_AGG across all balloon review PDFs in the stage, aggregating them into a single summary.

📌 **Note**: This example uses `AI_PARSE_DOCUMENT` to extract text from PDFs before summarizing. `AI_PARSE_DOCUMENT` will be covered in detail later in the Intelligent Document Processing lab.

In [ ]:
%%sql -r Summarize_Multiple_Docs_sql
-- Parse multiple PDFs from a stage and summarize all their content into one result
WITH AI_PARSE_DOC AS (
  SELECT
    RELATIVE_PATH AS file_name,
    GET_PRESIGNED_URL(
      @{{user}}_genai_db.resources.genai2day, 
      RELATIVE_PATH
    ) AS file_url,
    TO_VARCHAR(
      AI_PARSE_DOCUMENT(
        TO_FILE('@{{user}}_genai_db.resources.genai2day', RELATIVE_PATH),
        {'mode': 'LAYOUT', 'page_split': false}
      ):content
    ) AS raw_text
  FROM
    DIRECTORY(@{{user}}_genai_db.resources.genai2day)
  WHERE
    RELATIVE_PATH LIKE 'source_files/balloon_review%.pdf'
)
SELECT
  AI_SUMMARIZE_AGG(raw_text) AS balloon_summary
FROM
  AI_PARSE_DOC;

In [ ]:
import textwrap

# Convert the SQL result to a pandas DataFrame and display the balloon review summary
df = Summarize_Multiple_Docs_sql.to_pandas()
run_output = df["BALLOON_SUMMARY"].iloc[0]
wrapped_text = "\n".join(textwrap.wrap(run_output, width=80))
print(wrapped_text)

## Work With The Python API To Access Cortex LLM Functions

Cortex LLM functions are also callable from Snowpark Python using `call_udf` - the same functions used in SQL can be applied to DataFrame columns.

### Call Cortex LLM functions from Snowpark Python.

The example below calls `snowflake.cortex.Summarize` and `snowflake.cortex.Translate` on the review table using Snowpark Python.

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col, call_udf

# Use the Snowpark Python API to call Cortex AI functions on DataFrame columns
table_name = user + '_genai_db.raw.review'

def main(session: snowpark.Session): 
    # Load the review table as a Snowpark DataFrame
    activity_review_df = session.table(table_name)
    
    # Add summary and Japanese translation columns using Cortex UDFs
    activity_review_df = activity_review_df.with_columns(
        ["summary", "translation_to_japanese"],
        [call_udf("snowflake.cortex.Summarize", col("review_text")), 
        call_udf("snowflake.cortex.Translate", col("review_text"), "en", "ja")]
    )

    # Preview the first 2 rows
    activity_review_df.show(2)
    
    # Execute the full query
    activity_review_df.collect()
    
    return activity_review_df

main(session)

## AI_CLASSIFY

Classifies text into categories you define and returns a JSON object with the predicted label and confidence score. Supports multi-label output and accepts optional label descriptions and examples.

### AI_CLASSIFY use cases.

| Industry | Use Case | Business Value |
| :--- | :--- | :--- |
| **SaaS & Customer Service** 🎧 | **Automated Ticket Routing** | **Accelerates issue resolution** by automatically classifying and routing incoming support tickets (e.g., 'Billing', 'Technical Issue') to the correct team. |
| **E-commerce & Retail** 🛒 | **Customer Intent Recognition** | **Enhances customer self-service** by classifying user queries (e.g., 'Track Order', 'Return Item') to direct them to the right information faster. |
| **Sales & Marketing** 🎯 | **Sales Lead Qualification** | **Increases sales productivity** by automatically classifying inbound leads as 'High Priority' or 'Nurture' based on the content of their inquiry. |

### Classifying free text.

Classify a simple phrase into one of two categories to see the JSON output format AI_CLASSIFY returns.

In [ ]:
%%sql -r Classify_Free_Text_sql
-- Classify free text into one of the provided categories
SELECT AI_CLASSIFY('free the mind', ['yoga', 'exercise']);

### Classify traveler reviews.

Use AI_CLASSIFY to categorize negative reviews as `equipment` or `service` issues - useful for routing complaints to the right team.

In [ ]:
%%sql -r Classify_Reviews_sql
-- Classify negative reviews into 'equipment' or 'service' categories
SELECT review_text, AI_CLASSIFY(review_text, ['equipment','service']) AS classification 
FROM {{user}}_genai_db.presentation.traveler_activity 
WHERE snowflake.cortex.sentiment(review_text) <= -0.075;

### Classify customer messages.

Classify email and message intent as `refund` or `exchange` to demonstrate how AI_CLASSIFY can automate routing of incoming customer communications.

In [ ]:
%%sql -r Classify_Messages_sql
-- Classify customer messages as either 'refund' or 'exchange' requests
SELECT 
   AI_CLASSIFY('I would like to request a refund for my flight that was canceled last week. Please let me know the process.', ['refund', 'exchange']) AS classify_message_1,
  AI_CLASSIFY('Can I exchange my hotel reservation for different dates? The current ones no longer work for my schedule.', ['refund', 'exchange']) AS classify_message_2,
   AI_CLASSIFY('I am disappointed with my recent trip and would like to discuss a refund for the services that did not meet my expectations.', ['refund', 'exchange']) AS classify_message_3,
   AI_CLASSIFY('I need to change my booking to a later date due to unforeseen circumstances. How can I initiate the exchange?', ['refund', 'exchange']) AS classify_message_4,
   AI_CLASSIFY('I received the wrong tickets for my tour and would like a refund or a replacement, please assist me with this issue.', ['refund', 'exchange']) AS classify_message_5;

### Classify lost opportunities.

Categorize lost sales notes into `cost`, `marketing`, or `staff` to identify the root cause of each missed booking.

In [ ]:
%%sql -r Classify_Lost_Opportunities_sql
-- Classify lost business opportunities into 'cost', 'marketing', or 'staff' root causes
SELECT 
   AI_CLASSIFY('Our prices were higher than competitors, causing customers to choose other options.', ['cost', 'marketing', 'staff']) AS classify_opportunity_1,
    AI_CLASSIFY('Our website lacked engaging content and clear call-to-action, resulting in lost inquiries.', ['cost', 'marketing', 'staff']) AS classify_opportunity_2,
   AI_CLASSIFY('We did not respond quickly enough to customer inquiries, leading to missed bookings.', ['cost', 'marketing', 'staff']) AS classify_opportunity_3,
   AI_CLASSIFY('Our advertising efforts were not reaching the target audience effectively, causing low engagement.', ['cost', 'marketing', 'staff']) AS classify_opportunity_4,
    AI_CLASSIFY('Insufficient training for staff resulted in poor customer service experiences, driving clients away.', ['cost', 'marketing', 'staff']) AS classify_opportunity_5;

### Using AI_CLASSIFY with parameters.

Pass `task_description`, `output_mode: 'multi'`, and `examples` in the options object to enable multi-label classification - the model can return more than one label when several categories apply.

In [ ]:
%%sql -r Classify_Parameters_sql
-- This query classifies the input text into multiple relevant topics.
SELECT AI_CLASSIFY(
  -- Argument 1: The input text to be classified.
  'It is windy and cold outside. The rain will come soon. Our travel plans will be messed up. Traffic will build up too',

  -- Argument 2: A list of potential labels (categories) for classification.
  -- The 'description' for 'travel' provides extra context for the model.
  [
    {'label': 'travel', 'description': 'content related to traveling'},
    {'label': 'weather'},
    {'label': 'reading'},
    {'label': 'driving'}
  ],

  -- Argument 3: A JSON object containing options to configure the model's behavior.
  {
    -- Specifies the overall goal for the AI model.
    'task_description': 'Determine topics related to the given text',

    -- Allows the model to return multiple labels if more than one applies.
    'output_mode': 'multi',

    -- Provides a "few-shot" example to guide the model's reasoning and output format.
    'examples': [
      {
        'input': 'i love listening to music while traveling',
        'labels': ['travel', 'music'],
        'explanation': 'the text mentions traveling and listening to music'
      }
    ]
  }
);

## AI_EMBED

Converts text into a list of numbers (called a "vector") that represents its meaning. Think of it as GPS coordinates for language - text with similar meaning gets similar coordinates, even when the words are completely different.

This is the foundation for "search by meaning" instead of "search by keywords." Once you have embeddings, you can find related content, cluster similar documents, or build recommendation systems.

### AI_EMBED use cases.

| Industry | Use Case | Business Value |
| :--- | :--- | :--- |
| **E-commerce** 🛒 | **Product Recommendation** | Embed product descriptions and find semantically similar items to power recommendation features. |
| **Customer Support** 📞 | **Ticket Deduplication** | Embed support tickets and cluster similar issues to identify recurring problems and route efficiently. |
| **Legal & Compliance** ⚖️ | **Document Similarity Search** | Embed contract clauses to quickly find semantically similar provisions across large document libraries. |

### Generate embeddings for traveler reviews.

Run the query below to convert a traveler review into an embedding vector. The output is a long list of numbers - each review gets its own unique "fingerprint" based on what it means, not what words it uses.

In [ ]:
%%sql -r Generate_embeddings_sql
-- Generate an embedding for a sample review text
SELECT AI_EMBED(
    'snowflake-arctic-embed-m-v1.5',
    'The balloon ride was absolutely breathtaking. Stunning views and a very professional team.'
) AS review_embedding;

## AI_SIMILARITY

Measures how similar two pieces of text are in meaning, returning a score from 0 (completely unrelated) to 1 (same meaning). Unlike keyword matching, this understands that "amazing balloon ride" and "fantastic hot air experience" are talking about the same thing.

Use it to find the closest match to a search query, detect near-duplicate content, or rank results by relevance.

### AI_SIMILARITY use cases.

| Industry | Use Case | Business Value |
| :--- | :--- | :--- |
| **Media & Publishing** 📰 | **Duplicate Content Detection** | Score article similarity to flag near-duplicate submissions and maintain content uniqueness. |
| **HR & Recruiting** 👩‍💼 | **Resume-to-Job Matching** | Compare candidate profiles to job descriptions semantically to surface best-fit candidates. |
| **Travel & Hospitality** ✈️ | **Review Deduplication** | Identify suspiciously similar reviews to detect spam or copy-paste fraud. |

### Find the most similar reviews in the review table.

Given a target phrase, rank every review in the table by how closely it matches in meaning. Reviews about similar topics (views, guides, balloon experiences) will score highest - even if they use different words.

In [ ]:
%%sql -r Find_similar_reviews_sql
-- Find the 5 most semantically similar reviews to a given input
SET target_review = 'spectacular views from the balloon, excellent guide very knowledgeable';

SELECT
    review_text,
    ROUND(AI_SIMILARITY(
        $target_review,
        review_text,
        {'model': 'snowflake-arctic-embed-m-v1.5'}
    ), 4) AS similarity_score
FROM {{user}}_genai_db.raw.review
ORDER BY similarity_score DESC
LIMIT 5;

## 🎯 Challenge Questions

Test your understanding of the concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "What type of value does the AI_SENTIMENT function return?", "options": ["A) A text label such as positive, negative, or neutral", "B) AI_SENTIMENT returns a numeric value between -1 and 1", "C) A JSON object with sentiment categories", "D) A boolean true or false value"], "hash": "a01af30caf192f091e53bdee878acfbb"},
    {"q": "Which statement about AI_SUMMARIZE_AGG is TRUE?", "options": ["A) AI_SUMMARIZE_AGG requires you to specify the maximum number of words in the summary", "B) AI_SUMMARIZE_AGG is an aggregate function that can handle datasets larger than a single model context window", "C) AI_SUMMARIZE_AGG can only summarize data stored in Snowflake tables", "D) AI_SUMMARIZE_AGG always returns exactly 100 words"], "hash": "f4b15bb5b897cce257a8d9d3cc0a888b"},
    {"q": "What makes AI_SUMMARIZE_AGG different from a simple summarization function?", "options": ["A) It can only process one document at a time", "B) AI_SUMMARIZE_AGG is an aggregate function that combines multiple text inputs", "C) It requires a pre-trained custom model", "D) It only works with structured data in tables"], "hash": "7ea280d2b1e6dc8d8b9245aa597bf6a2"},
    {"q": "Which capability does AI_CLASSIFY provide?", "options": ["A) It can only classify text into two categories", "B) It requires training data before use", "C) It only works with English language text", "D) AI_CLASSIFY can assign multiple categories to a single input"], "hash": None, "correct": "D"},
    {"q": "What does the AI_TRANSLATE function do when an empty string is passed as the source language code?", "options": ["A) It returns an error because a source language code is required", "B) It defaults to English as the source language", "C) AI_TRANSLATE automatically detects the source language when an empty string is passed", "D) It skips translation and returns the original text unchanged"], "hash": "df81c64d23e3a85387128ad0ea77b2b3"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        if item["hash"] is None:
            # Hardcoded validation for updated question
            is_correct = (letter == item["correct"])
            feedback = "\u2705 Correct!" if is_correct else "\u26d4 Incorrect. Try again"
        else:
            escaped_opt = opt.replace("'", "''")
            result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
            feedback = result[0][0]
            is_correct = 'Correct' in feedback or '\u2705' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))


## Key Takeaways

❄️ Snowflake Cortex AI provides task-based functions that bring AI to your data with a single SQL call - no model management, no infrastructure.

❄️ Each function is purpose-built: SENTIMENT scores tone, AI_SENTIMENT breaks it down by category, AI_TRANSLATE handles multilingual data, AI_SUMMARIZE_AGG condenses large volumes of text, and AI_CLASSIFY routes content into your defined categories.

❄️ **AI_EMBED and AI_SIMILARITY** enable semantic search and similarity scoring - generate embeddings to represent text as vectors, then compare them to find related or near-duplicate content.

❄️ Data quality directly impacts function output - assess your data before running AI functions and be mindful of sensitive content.

❄️ All Cortex AI functions are accessible from both SQL and Snowpark Python via `call_udf`, enabling integration into data pipelines.